In [1]:
# https://gml.noaa.gov/aftp/data/radiation/baseline/spo
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.dates as mdates
import os
import urllib.request
import random
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from numpy import trapezoid
from scipy.signal import argrelmin

# Global Variables
Area = 2.0 #m**2
baseurl = 'https://gml.noaa.gov/aftp/data/radiation/baseline/spo'
InputPath = '../../DataFolder'
OutputPath = 'PlotsFolder/'

In [2]:
# Lets adapt our model to include zenith angle

def model(time, Direct_Irradiance, Diffuse_Irradiance, Upwelling_Irradiance, effs, sharpness, panel_angle_negative, zenith_angle, solar_zenith_angle):
    '''
    Models solar pnal output curve for one day based on NOAA ARO SPO GML Irradiance data
    This model function is like the old one but lets you put in the angle of the solar panel
    '''
    eff_direct = effs[0]
    eff_isotropic = effs[1]
    efficiency_back = .7

    # The angle the math uses to calculate is not the angle which makes sens for us to define our panel angle as so I correct for it as such
    angle_panel = 360 - panel_angle_negative
    Change_Time_1 = angle_panel/360*1440
    Change_Time_2 = Change_Time_1 + 1440/2
    Angular_Dependance_Front = np.array(np.cos(np.pi/(Change_Time_2 - Change_Time_1)*(time - (Change_Time_1 + Change_Time_2)/2)))
    Angular_Dependance_Back = np.array(-Angular_Dependance_Front)

    # create front of panel simulation (I did tricky math to get these)
    # Directional components:
    Front = eff_direct*Direct_Irradiance*Angular_Dependance_Front*sigmoid(time, sharpness, -Change_Time_1)*sigmoid(-time, sharpness, Change_Time_2)
    Back  = .7*eff_direct*Direct_Irradiance*Angular_Dependance_Back*sigmoid(time, sharpness, -Change_Time_2)*sigmoid(-time, sharpness, 2*Change_Time_2 - Change_Time_1)
    Secondary_Front = eff_direct*Direct_Irradiance*Angular_Dependance_Front*sigmoid(time, sharpness, Change_Time_1 - 2*Change_Time_2)*sigmoid(-time, sharpness, 3*Change_Time_2 - Change_Time_1)
    Secondary_Back = .7*eff_direct*Direct_Irradiance*Angular_Dependance_Back*sigmoid(time, sharpness, Change_Time_2 - 2*Change_Time_1)*sigmoid(-time, sharpness, Change_Time_1)
    Tertiary_Front = eff_direct*Direct_Irradiance*Angular_Dependance_Front*sigmoid(time, sharpness, -Change_Time_1-2*Change_Time_1+2*Change_Time_2)*sigmoid(-time, sharpness, Change_Time_2-(-2*Change_Time_1+2*Change_Time_2))
    Tertiary_Back = .7*eff_direct*Direct_Irradiance*Angular_Dependance_Back*sigmoid(time, sharpness, -Change_Time_2+4*Change_Time_2-4*Change_Time_1)*sigmoid(-time, sharpness, 2*Change_Time_2 - Change_Time_1-(+4*Change_Time_2-4*Change_Time_1))
    # Total_Directional_Irradiance = Front + Back + Secondary_Front + Secondary_Back
    Total_Directional_Irradiance = Front + Back + Secondary_Front + Secondary_Back + Tertiary_Back + Tertiary_Front
    
    
    # add directional to the isotropic components:
    Total_Irradiance = Total_Directional_Irradiance + eff_isotropic*Diffuse_Irradiance + eff_isotropic*Upwelling_Irradiance

    # Multiply by area to get total power
    Total_Power = Total_Irradiance*Area

    # constraint of device resistor:

    Max_Power = 420
    Total_Power = np.clip(Total_Power, 0, Max_Power)
    return Total_Power

In [ ]:
# We have solved for the conversion factors for vertical panels in our vertical panel model notebooks. But these conversion factors are specifically for the angle between the sun and the panel when the panel is in the vertical orientation for the zenith angle the sun was at.
# We must then solve for a more universial conversion factor by deviding by cos(angle between panel normal vector and sun)
# We can do this because Power = ε_u * cos(angle between sun and panel) * Irradiance * Area of panel

# I have done this in Analysis/Model/Every_Day_Model.ipynb because that is where I do mutli day fitted statistics. I will copy the mean values for Direct and Isotropic over here





In [ ]:









def Horizontal_Power(time, Direct_Irradiance, Diffuse_Irradaince, Upwelling_Irradiance, effs, solar_zenith_angle):
    '''this function is simulates the power output curve for a horizontal (laying flat) panel based on NOAA ARO SPO GML irradiance data'''


    









    